# Trace Analysis

In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display  # noqa: A004
from trace_viewer import show_trace

TRACES_DIR = Path("../traces")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 50)

## Dashboard

In [ ]:
raw = pd.read_csv(TRACES_DIR / "results.csv")
raw["passed"] = raw["passed"].astype(bool)
for col in ["model_time_s", "tool_time_s"]:
    if col not in raw.columns:
        raw[col] = float("nan")
if "run_type" not in raw.columns:
    raw["run_type"] = ""

# Keep only the latest row per (model, trace_id).
# Reruns and full runs both append to the CSV chronologically,
# so the last entry is always the most current.
results = (
    raw.sort_values("timestamp")
    .drop_duplicates(subset=["model", "trace_id"], keep="last")
    .reset_index(drop=True)
)
print(f"{len(results)} results across {results['model'].nunique()} models")

In [ ]:
# Run-over-run summary (full runs only, last 10)
full_runs = results[results["run_type"].isin(["full", ""])]
runs = (
    full_runs.groupby(["timestamp", "model"])
    .agg(
        passed=("passed", "sum"),
        total=("passed", "count"),
        pass_rate=("passed", "mean"),
        avg_calls=("tool_calls", "mean"),
        avg_wall=("wall_time_s", "mean"),
        avg_tokens=("total_tokens", "mean"),
        total_cost=("cost_usd", "sum"),
    )
)
# Drop error rows (0 questions passed, 0 cost)
runs = runs[runs["total"] > 2].tail(10)

runs["pass_rate"] = runs["pass_rate"].map("{:.0%}".format)
runs["result"] = runs["passed"].astype(int).astype(str) + "/" + runs["total"].astype(str)
runs["avg_calls"] = runs["avg_calls"].round(1)
runs["avg_wall"] = runs["avg_wall"].map("{:.0f}s".format)
runs["avg_tokens"] = runs["avg_tokens"].map("{:,.0f}".format)
runs["total_cost"] = runs["total_cost"].map("${:.2f}".format)

# Clean up model names and index
runs = runs.reset_index()
runs["model"] = runs["model"].str.replace("openrouter/", "").str.replace("google/", "").str.replace("qwen/", "")
runs["date"] = runs["timestamp"].str[:10]

runs[["date", "model", "result", "pass_rate", "avg_calls", "avg_wall", "avg_tokens", "total_cost"]]

In [ ]:
# Select a run to inspect (default: latest)
RUN = results["timestamp"].max()

run = results[results["timestamp"] == RUN]
passed = run["passed"].sum()
total = len(run)
print(f"Selected: {RUN} | {passed}/{total} passed")

run_cols = [
    "workspace",
    "category",
    "question",
    "passed",
    "tool_calls",
    "scratchpad_files",
    "total_tokens",
    "cost_usd",
    "model_time_s",
    "wall_time_s",
    "failed_assertions",
]
# scratchpad_files may not exist in older CSVs
for col in run_cols:
    if col not in run.columns:
        run[col] = ""
run.sort_values("passed")[run_cols]

In [ ]:
# Failures for the selected run
failures = run[~run["passed"]]
if len(failures):
    print(f"{len(failures)} failures:")
    display(
        failures[["workspace", "question", "tool_calls", "failed_assertions", "error"]]
    )
else:
    print("No failures!")

## Trace Viewer

In [ ]:
# Build trace index: model → [(label, path), ...]
# Default to model with most recently modified trace file.
trace_index: dict[str, list[tuple[str, Path]]] = {}
model_recency: dict[str, float] = {}
for model_dir in sorted(TRACES_DIR.iterdir()):
    if not model_dir.is_dir() or model_dir.name.endswith(".csv"):
        continue
    entries = []
    latest_mtime = 0.0
    for ws_dir in sorted(model_dir.iterdir()):
        if not ws_dir.is_dir():
            continue
        for f in sorted(ws_dir.glob("*.json")):
            latest_mtime = max(latest_mtime, f.stat().st_mtime)
            data = json.loads(f.read_text())
            status = "PASS" if data["passed"] else "FAIL"
            entries.append((f"[{status}] {ws_dir.name} / {f.stem}", f))
    if entries:
        trace_index[model_dir.name] = entries
        model_recency[model_dir.name] = latest_mtime

latest_model = max(model_recency, key=model_recency.get)  # type: ignore[arg-type]


def _default_question(entries: list[tuple[str, Path]]) -> str:
    """Pick first failed question, or first question if all pass."""
    for label, _ in entries:
        if label.startswith("[FAIL]"):
            return label
    return entries[0][0]


model_dropdown = widgets.Dropdown(
    options=list(trace_index.keys()),
    value=latest_model,
    description="Model:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="60%"),
)
question_dropdown = widgets.Dropdown(
    description="Question:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="90%"),
)
output = widgets.Output()


def _update_questions(_change=None):
    entries = trace_index[model_dropdown.value]
    question_dropdown.options = [label for label, _ in entries]
    question_dropdown.value = _default_question(entries)


def _show_trace(_change=None):
    entries = trace_index[model_dropdown.value]
    label_to_path = dict(entries)
    path = label_to_path.get(question_dropdown.value)
    if not path:
        return
    data = json.loads(path.read_text())
    with output:
        output.clear_output(wait=True)
        show_trace(data, max_chars=2000)


model_dropdown.observe(_update_questions, names="value")
question_dropdown.observe(_show_trace, names="value")

_update_questions()
display(model_dropdown, question_dropdown, output)
_show_trace()